# [실습] 텍스트 전처리 핵심 기법

## 학습목표
- 정규표현식(Regular Expression)을 활용하여 텍스트를 처리할 수 있다
- 텍스트 정제, 토큰화, 정규화를 수행할 수 있다
- 불용어 제거와 형태소 분석을 적용할 수 있다
- 실제 뉴스 기사에 전처리 파이프라인을 구축할 수 있다

## 정규표현식(Regular Expression) 기초

정규표현식은 텍스트 패턴을 찾고 처리하는 강력한 도구입니다.  
NLP 전처리에서 필수적으로 사용됩니다.

In [2]:
import re
import pandas as pd
import json
from collections import Counter

# 정규표현식 기본 패턴
text = "연락처: 010-1234-5678, 이메일: user@example.com, 날짜: 2024-01-15"

# 전화번호 찾기
phone_pattern = r'\d{3}-\d{4}-\d{4}'
phones = re.findall(phone_pattern, text)
print(f"전화번호: {phones}")

# 이메일 찾기
email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
emails = re.findall(email_pattern, text)
print(f"이메일: {emails}")

# 날짜 찾기
date_pattern = r'\d{4}-\d{2}-\d{2}'
dates = re.findall(date_pattern, text)
print(f"날짜: {dates}")

전화번호: ['010-1234-5678']
이메일: ['user@example.com']
날짜: ['2024-01-15']


In [3]:
# 주요 정규표현식 패턴 실습
sample_text = """Python3.13 버전이 출시되었습니다! 
새로운 기능들: 1) 성능 향상 2) 타입 힌트 개선 3) 에러 메시지 개선
문의: support@python.org 또는 02-123-4567"""

patterns = {
    '숫자': r'\d+',
    '영문 단어': r'[a-zA-Z]+',
    '한글 단어': r'[가-힣]+',
    '괄호 안 내용': r'\([^)]*\)',
    '문장 부호': r'[!?.,;:]',
    '버전 번호': r'\d+\.\d+'
}

print("=== 정규표현식 패턴 매칭 결과 ===")
for name, pattern in patterns.items():
    matches = re.findall(pattern, sample_text)
    print(f"{name}: {matches[:5]}...")  # 처음 5개만 표시

=== 정규표현식 패턴 매칭 결과 ===
숫자: ['3', '13', '1', '2', '3']...
영문 단어: ['Python', 'support', 'python', 'org']...
한글 단어: ['버전이', '출시되었습니다', '새로운', '기능들', '성능']...
괄호 안 내용: []...
문장 부호: ['.', '!', ':', ':', '.']...
버전 번호: ['3.13']...


## 텍스트 정제(Cleaning)

불필요한 문자와 노이즈를 제거하는 과정입니다.

In [4]:
def clean_text(text):
    """텍스트 정제 함수"""
    # HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    
    # URL 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 이메일 제거
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)
    
    # 특수문자를 공백으로 치환 (한글, 영문, 숫자, 기본 문장부호만 유지)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', ' ', text)
    
    # 연속된 공백을 하나로
    text = re.sub(r'\s+', ' ', text)
    
    # 앞뒤 공백 제거
    text = text.strip()
    
    return text

# 테스트
messy_text = """<p>안녕하세요!</p> 웹사이트 https://example.com을 방문하세요.
문의: contact@example.com ☎️ 02-1234-5678 #AI #머신러닝"""

cleaned = clean_text(messy_text)
print("원본 텍스트:")
print(messy_text)
print("\n정제된 텍스트:")
print(cleaned)

원본 텍스트:
<p>안녕하세요!</p> 웹사이트 https://example.com을 방문하세요.
문의: contact@example.com ☎️ 02-1234-5678 #AI #머신러닝

정제된 텍스트:
안녕하세요! 웹사이트 방문하세요. 문의 02 1234 5678 AI 머신러닝


In [5]:
# 다양한 노이즈 처리
def remove_noise(text):
    """다양한 노이즈 제거"""
    # 이모지 제거
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    
    # 해시태그 처리 (태그만 추출)
    hashtags = re.findall(r'#\w+', text)
    text = re.sub(r'#\w+', '', text)
    
    # 멘션 제거
    text = re.sub(r'@\w+', '', text)
    
    return text, hashtags

social_text = "오늘 날씨 좋네요! 😊 #날씨 #맑음 @friend1 과 함께 🌸"
cleaned_social, tags = remove_noise(social_text)
print(f"원본: {social_text}")
print(f"정제: {cleaned_social}")
print(f"해시태그: {tags}")

원본: 오늘 날씨 좋네요! 😊 #날씨 #맑음 @friend1 과 함께 🌸
정제: 오늘 날씨 좋네요!     과 함께 
해시태그: ['#날씨', '#맑음']


## 토큰화(Tokenization)

텍스트를 의미 있는 단위로 분할하는 과정입니다.

In [6]:
def tokenize_korean(text):
    """한국어 토큰화 (간단한 공백 기반)"""
    # 문장 분리
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    # 단어 분리
    words = text.split()
    
    # 형태소 단위 (간단한 조사 분리)
    morphemes = []
    josa_pattern = r'(은|는|이|가|을|를|에|에서|로|으로|와|과|의|도|만|부터|까지)$'
    
    for word in words:
        match = re.search(josa_pattern, word)
        if match:
            stem = word[:match.start()]
            josa = match.group()
            if stem:
                morphemes.append(stem)
            morphemes.append(josa)
        else:
            morphemes.append(word)
    
    return {
        'sentences': sentences,
        'words': words,
        'morphemes': morphemes
    }

korean_text = "자연어처리는 인공지능의 한 분야입니다. 텍스트를 컴퓨터가 이해할 수 있도록 처리합니다."
tokens = tokenize_korean(korean_text)

print("=== 토큰화 결과 ===")
print(f"문장: {tokens['sentences']}")
print(f"단어: {tokens['words']}")
print(f"형태소: {tokens['morphemes']}")

=== 토큰화 결과 ===
문장: ['자연어처리는 인공지능의 한 분야입니다', '텍스트를 컴퓨터가 이해할 수 있도록 처리합니다']
단어: ['자연어처리는', '인공지능의', '한', '분야입니다.', '텍스트를', '컴퓨터가', '이해할', '수', '있도록', '처리합니다.']
형태소: ['자연어처리', '는', '인공지능', '의', '한', '분야입니다.', '텍스트', '를', '컴퓨터', '가', '이해할', '수', '있도록', '처리합니다.']


In [7]:
# N-gram 토큰화
def get_ngrams(text, n=2):
    """N-gram 토큰 생성"""
    words = text.split()
    ngrams = []
    
    for i in range(len(words) - n + 1):
        ngram = ' '.join(words[i:i+n])
        ngrams.append(ngram)
    
    return ngrams

text = "딥러닝 모델을 학습시키기 위해 데이터를 전처리합니다"

print("=== N-gram 토큰화 ===")
for n in [1, 2, 3]:
    ngrams = get_ngrams(text, n)
    print(f"{n}-gram: {ngrams}")

=== N-gram 토큰화 ===
1-gram: ['딥러닝', '모델을', '학습시키기', '위해', '데이터를', '전처리합니다']
2-gram: ['딥러닝 모델을', '모델을 학습시키기', '학습시키기 위해', '위해 데이터를', '데이터를 전처리합니다']
3-gram: ['딥러닝 모델을 학습시키기', '모델을 학습시키기 위해', '학습시키기 위해 데이터를', '위해 데이터를 전처리합니다']


## 정규화(Normalization)

텍스트를 표준 형태로 변환하는 과정입니다.

In [8]:
def normalize_text(text):
    """텍스트 정규화"""
    # 대소문자 통일 (영문)
    text = text.lower()
    
    # 숫자 정규화
    text = re.sub(r'\d+', 'NUM', text)
    
    # 반복 문자 축소
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 공백 정규화
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# 테스트
texts = [
    "Python3와 Python2는 다릅니다!!!!!!",
    "와아아아아아 대박!!!!   너무    좋아요",
    "가격은 15,000원 입니다. 2024년 1월 출시"
]

print("=== 정규화 결과 ===")
for text in texts:
    normalized = normalize_text(text)
    print(f"원본: {text}")
    print(f"정규화: {normalized}\n")

=== 정규화 결과 ===
원본: Python3와 Python2는 다릅니다!!!!!!
정규화: pythonNUM와 pythonNUM는 다릅니다!!

원본: 와아아아아아 대박!!!!   너무    좋아요
정규화: 와아아 대박!! 너무 좋아요

원본: 가격은 15,000원 입니다. 2024년 1월 출시
정규화: 가격은 NUM,NUM원 입니다. NUM년 NUM월 출시



In [9]:
# 한글 자모 분리/결합 (간단한 예시)
def normalize_korean_chars(text):
    """한글 문자 정규화"""
    # ㅋㅋㅋ, ㅎㅎㅎ 같은 자음 반복 정규화
    text = re.sub(r'[ㅋ]{2,}', 'ㅋㅋ', text)
    text = re.sub(r'[ㅎ]{2,}', 'ㅎㅎ', text)
    text = re.sub(r'[ㅠㅜ]{2,}', 'ㅠㅠ', text)
    
    # 이상한 띄어쓰기 교정 (간단한 규칙)
    text = re.sub(r'\s+([.,!?])', r'\1', text)  # 문장부호 앞 공백 제거
    
    return text

korean_samples = [
    "ㅋㅋㅋㅋㅋㅋㅋ 너무 웃겨요",
    "ㅠㅠㅠㅠㅠ 슬퍼요",
    "안녕하세요 . 반갑습니다 !"
]

for sample in korean_samples:
    normalized = normalize_korean_chars(sample)
    print(f"원본: {sample}")
    print(f"정규화: {normalized}\n")

원본: ㅋㅋㅋㅋㅋㅋㅋ 너무 웃겨요
정규화: ㅋㅋ 너무 웃겨요

원본: ㅠㅠㅠㅠㅠ 슬퍼요
정규화: ㅠㅠ 슬퍼요

원본: 안녕하세요 . 반갑습니다 !
정규화: 안녕하세요. 반갑습니다!



## 불용어(Stopwords) 처리

분석에 불필요한 단어들을 제거하는 과정입니다.

In [15]:
# 한국어 불용어 리스트
korean_stopwords = [
    '이', '있', '하', '것', '를', '들', '그', '되', '수', '이', '보', '않', '없', '나', '사람',
    '주', '아니', '등', '같', '우리', '때', '년', '가', '한', '지', '대하', '오', '말',
    '일', '그렇', '위하', '때문', '그것', '두', '말하', '알', '그러나', '받', '못하',
    '일', '그런', '또',  '더', '사회', '많', '그리고', '좋', '크', '따르', '중'
]

def remove_stopwords(text, stopwords):
    """불용어 제거"""
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords]
    return ' '.join(filtered_words)

# 테스트
text = "다음 은 매우 중요한 문제 입니다 우리 함께 문제 를 같이 해결해야 할 것 입니다"
filtered_text = remove_stopwords(text, korean_stopwords)

print(f"원본: {text}")
print(f"불용어 제거: {filtered_text}")
print(f"제거된 단어: {set(text.split()) - set(filtered_text.split())}")

원본: 다음 은 매우 중요한 문제 입니다 우리 함께 문제 를 같이 해결해야 할 것 입니다
불용어 제거: 다음 은 매우 중요한 문제 입니다 함께 문제 같이 해결해야 할 입니다
제거된 단어: {'것', '우리', '를'}


## 실전: 뉴스 기사 전처리 파이프라인

실제 뉴스 데이터에 전처리 파이프라인을 구축해보겠습니다.

In [14]:
# 뉴스 기사 데이터 로드
try:
    with open('raw_articles.txt', 'r', encoding='utf-8') as f:
        articles = f.read().split('\n---\n')
    print(f"총 {len(articles)}개 기사 로드")
except FileNotFoundError:
    # 샘플 데이터 생성
    articles = [
        """[속보] AI 기술 혁신으로 산업 전반 변화 예상
        전문가들은 인공지능(AI) 기술이 향후 5년 내 모든 산업에 큰 영향을 미칠 것으로 전망했다.
        특히 제조업과 서비스업에서의 자동화가 가속화될 것으로 보인다.""",
        
        """파이썬(Python), 개발자가 가장 배우고 싶은 언어 1위
        Stack Overflow 조사에 따르면 Python이 5년 연속 가장 인기 있는 프로그래밍 언어로 선정됐다.
        데이터 분석과 AI 개발에서의 활용도가 높은 것이 주요 요인으로 분석된다.""",
        
        """ChatGPT 사용자 2억명 돌파... AI 대중화 시대 열려
        OpenAI의 ChatGPT가 출시 1년 만에 사용자 2억명을 돌파했다고 발표했다.
        일상생활에서 AI를 활용하는 사례가 급증하고 있다."""
    ]
    print(f"샘플 데이터 {len(articles)}개 생성")

print("\n첫 번째 기사:")
print(articles[0][:200] + "...")

총 7개 기사 로드

첫 번째 기사:
[특집] 대규모 언어모델(LLM)이 바꾸는 미래
인공지능 기술의 핵심으로 떠오른 대규모 언어모델(Large Language Model, LLM)이 산업 전반에 혁명적인 변화를 일으키고 있다. OpenAI의 GPT-4, 구글의 Gemini, 메타의 LLaMA 등 다양한 모델들이 경쟁하며 기술 발전을 가속화하고 있다.
전문가들은 LLM이 단순한 텍스트 생성을 ...


In [16]:
class TextPreprocessor:
    """텍스트 전처리 파이프라인 클래스"""
    
    def __init__(self, stopwords=None):
        self.stopwords = stopwords or korean_stopwords
        self.stats = {}
    
    def preprocess(self, text):
        """전체 전처리 파이프라인"""
        # 원본 통계
        original_length = len(text)
        original_words = len(text.split())
        
        # 1. 제목과 본문 분리
        lines = text.strip().split('\n')
        title = lines[0] if lines else ''
        body = ' '.join(lines[1:]) if len(lines) > 1 else ''
        
        # 2. 정제
        title = self._clean(title)
        body = self._clean(body)
        
        # 3. 토큰화
        title_tokens = self._tokenize(title)
        body_tokens = self._tokenize(body)
        
        # 4. 정규화
        title_normalized = self._normalize(title)
        body_normalized = self._normalize(body)
        
        # 5. 불용어 제거
        title_filtered = self._remove_stopwords(title_normalized)
        body_filtered = self._remove_stopwords(body_normalized)
        
        # 처리 후 통계
        processed_length = len(title_filtered) + len(body_filtered)
        processed_words = len(title_filtered.split()) + len(body_filtered.split())
        
        return {
            'title': title_filtered,
            'body': body_filtered,
            'title_tokens': title_tokens,
            'body_tokens': body_tokens,
            'stats': {
                'original_length': original_length,
                'processed_length': processed_length,
                'original_words': original_words,
                'processed_words': processed_words,
                'reduction_rate': 1 - (processed_words / original_words) if original_words > 0 else 0
            }
        }
    
    def _clean(self, text):
        """텍스트 정제"""
        text = re.sub(r'\[.*?\]', '', text)  # 대괄호 내용 제거
        text = re.sub(r'\(.*?\)', '', text)  # 소괄호 내용 제거
        text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?%]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def _tokenize(self, text):
        """토큰화"""
        return text.split()
    
    def _normalize(self, text):
        """정규화"""
        text = text.lower()
        text = re.sub(r'\d+', 'NUM', text)
        return text
    
    def _remove_stopwords(self, text):
        """불용어 제거"""
        words = text.split()
        filtered = [w for w in words if w not in self.stopwords]
        return ' '.join(filtered)

# 전처리기 생성 및 적용
preprocessor = TextPreprocessor()

print("=== 기사 전처리 결과 ===")
for i, article in enumerate(articles, 1):
    result = preprocessor.preprocess(article)
    print(f"\n기사 {i}:")
    print(f"제목: {result['title']}")
    print(f"본문 (앞 100자): {result['body'][:100]}...")
    print(f"통계: 원본 {result['stats']['original_words']}단어 → "
          f"처리 후 {result['stats']['processed_words']}단어 "
          f"(감소율: {result['stats']['reduction_rate']:.1%})")

=== 기사 전처리 결과 ===

기사 1:
제목: 대규모 언어모델이 바꾸는 미래
본문 (앞 100자): 인공지능 기술의 핵심으로 떠오른 대규모 언어모델이 산업 전반에 혁명적인 변화를 일으키고 있다. openai의 gpt NUM, 구글의 gemini, 메타의 llama 다양한 모델들이...
통계: 원본 91단어 → 처리 후 83단어 (감소율: 8.8%)

기사 2:
제목: 삼성전자, ai 반도체 mach NUM 공개... nvidia와 경쟁 본격화
본문 (앞 100자): 삼성전자가 자체 개발한 ai 가속기 칩 mach NUM 을 공개했다. 칩은 최신 hbmNUMe 메모리를 탑재하고, 전력 효율을 기존 대비 NUM% 개선했다고 회사 측은 밝혔다. m...
통계: 원본 91단어 → 처리 후 94단어 (감소율: -3.3%)

기사 3:
제목: python NUM.NUM 정식 출시... jit 컴파일러로 성능 혁신
본문 (앞 100자): python software foundation이 python NUM.NUM을 정식 출시했다. 이번 버전의 가장 큰 특징은 실험적 jit 컴파일러 도입이다. jit 컴파일러는 자주...
통계: 원본 97단어 → 처리 후 95단어 (감소율: 2.1%)

기사 4:
제목: 제로데이 취약점 급증... 사이버 보안 패러다임 전환 필요
본문 (앞 100자): NUM년 상반기에만 제로데이 취약점이 전년 대비 NUM% 증가한 것으로 나타났다. 특히 ai를 활용한 자동화된 공격이 늘어나면서 기존 보안 체계의 한계가 드러나고 있다. 최근 발견...
통계: 원본 91단어 → 처리 후 95단어 (감소율: -4.4%)

기사 5:
제목: ai 코딩 어시스턴트 코드메이트 , 시리즈b NUM억원 투자 유치
본문 (앞 100자): 국내 ai 스타트업 코드메이트가 시리즈b 라운드에서 NUM억원 규모의 투자를 유치했다. 이번 투자는 소프트뱅크 벤처스가 리드하고, 기존 투자자인 네이버 dNUMsf, 카카오벤처스가...
통계: 원본 88단어 → 처리 후 88단어 (감

## 단어 빈도 분석

전처리된 텍스트에서 중요 키워드를 추출해보겠습니다.

In [17]:
def extract_keywords(texts, top_n=10):
    """텍스트에서 상위 키워드 추출"""
    all_words = []
    
    for text in texts:
        # 전처리
        processed = preprocessor.preprocess(text)
        words = processed['body'].split() + processed['title'].split()
        all_words.extend(words)
    
    # 단어 빈도 계산
    word_freq = Counter(all_words)
    
    # 영문과 한글 분리
    korean_words = {}
    english_words = {}
    
    for word, freq in word_freq.items():
        if re.match(r'[가-힣]+', word):
            korean_words[word] = freq
        elif re.match(r'[a-zA-Z]+', word):
            english_words[word] = freq
    
    return {
        'korean': sorted(korean_words.items(), key=lambda x: x[1], reverse=True)[:top_n],
        'english': sorted(english_words.items(), key=lambda x: x[1], reverse=True)[:top_n],
        'total': word_freq.most_common(top_n)
    }

keywords = extract_keywords(articles, top_n=5)

print("=== 주요 키워드 ===")
print("\n한글 키워드:")
for word, freq in keywords['korean']:
    print(f"  {word}: {freq}회")

print("\n영문 키워드:")
for word, freq in keywords['english']:
    print(f"  {word}: {freq}회")

=== 주요 키워드 ===

한글 키워드:
  특히: 9회
  있다.: 7회
  더욱: 5회
  시장: 4회
  보안: 4회

영문 키워드:
  ai: 14회
  NUM: 5회
  NUM%: 4회
  python: 4회
  mach: 3회


## 전처리 성능 비교

다양한 전처리 수준에 따른 결과를 비교해보겠습니다.

In [18]:
def compare_preprocessing_levels(text):
    """전처리 수준별 비교"""
    results = {}
    
    # Level 0: 원본
    results['원본'] = {
        'text': text,
        'length': len(text),
        'words': len(text.split())
    }
    
    # Level 1: 정제만
    cleaned = clean_text(text)
    results['정제'] = {
        'text': cleaned,
        'length': len(cleaned),
        'words': len(cleaned.split())
    }
    
    # Level 2: 정제 + 정규화
    normalized = normalize_text(cleaned)
    results['정제+정규화'] = {
        'text': normalized,
        'length': len(normalized),
        'words': len(normalized.split())
    }
    
    # Level 3: 정제 + 정규화 + 불용어
    filtered = remove_stopwords(normalized, korean_stopwords)
    results['전체처리'] = {
        'text': filtered,
        'length': len(filtered),
        'words': len(filtered.split())
    }
    
    return results

# 테스트
test_article = articles[0]
comparison = compare_preprocessing_levels(test_article)

print("=== 전처리 수준별 비교 ===")
for level, result in comparison.items():
    print(f"\n[{level}]")
    print(f"길이: {result['length']} 문자, {result['words']} 단어")
    print(f"텍스트 (앞 80자): {result['text'][:80]}...")

# 감소율 계산
original_words = comparison['원본']['words']
final_words = comparison['전체처리']['words']
reduction = (1 - final_words/original_words) * 100
print(f"\n총 단어 감소율: {reduction:.1f}%")

=== 전처리 수준별 비교 ===

[원본]
길이: 447 문자, 91 단어
텍스트 (앞 80자): [특집] 대규모 언어모델(LLM)이 바꾸는 미래
인공지능 기술의 핵심으로 떠오른 대규모 언어모델(Large Language Model, LLM)...

[정제]
길이: 444 문자, 100 단어
텍스트 (앞 80자): 특집 대규모 언어모델 LLM 이 바꾸는 미래 인공지능 기술의 핵심으로 떠오른 대규모 언어모델 Large Language Model, LLM 이 ...

[정제+정규화]
길이: 446 문자, 100 단어
텍스트 (앞 80자): 특집 대규모 언어모델 llm 이 바꾸는 미래 인공지능 기술의 핵심으로 떠오른 대규모 언어모델 large language model, llm 이 ...

[전체처리]
길이: 432 문자, 94 단어
텍스트 (앞 80자): 특집 대규모 언어모델 llm 바꾸는 미래 인공지능 기술의 핵심으로 떠오른 대규모 언어모델 large language model, llm 산업 전...

총 단어 감소율: -3.3%


## 실습 프로젝트: 뉴스 분류를 위한 전처리

카테고리별 뉴스 전처리 및 특징 추출을 수행합니다.

In [19]:
# 카테고리별 뉴스 데이터
categorized_news = {
    'AI': [
        "인공지능이 의료 진단에 혁명을 일으키고 있다. 딥러닝 기술로 암 진단 정확도 95% 달성",
        "ChatGPT 경쟁 심화, 구글과 메타도 대화형 AI 출시 준비"
    ],
    '개발': [
        "자바스크립트 프레임워크 React 18 출시, 성능 대폭 개선",
        "파이썬 웹 프레임워크 Django 5.0 베타 버전 공개"
    ],
    '보안': [
        "대규모 랜섬웨어 공격 증가, 기업들 보안 강화 나서",
        "제로데이 취약점 발견, 긴급 보안 패치 배포"
    ]
}

def analyze_category_features(news_dict):
    """카테고리별 특징 분석"""
    category_features = {}
    
    for category, articles in news_dict.items():
        all_text = ' '.join(articles)
        processed = preprocessor.preprocess(all_text)
        
        # 주요 키워드 추출
        words = processed['body'].split() + processed['title'].split()
        word_freq = Counter(words)
        top_keywords = word_freq.most_common(5)
        
        # 특수 패턴 찾기
        tech_terms = re.findall(r'[A-Z][a-z]+(?:[A-Z][a-z]+)*', all_text)  # CamelCase
        versions = re.findall(r'\d+\.\d+', all_text)  # 버전 번호
        percentages = re.findall(r'\d+%', all_text)  # 퍼센트
        
        category_features[category] = {
            'keywords': top_keywords,
            'tech_terms': list(set(tech_terms)),
            'versions': versions,
            'percentages': percentages,
            'avg_length': sum(len(a) for a in articles) / len(articles)
        }
    
    return category_features

features = analyze_category_features(categorized_news)

print("=== 카테고리별 특징 분석 ===")
for category, feature in features.items():
    print(f"\n[{category}]")
    print(f"주요 키워드: {[w for w, f in feature['keywords'][:3]]}")
    print(f"기술 용어: {feature['tech_terms'][:3]}")
    print(f"버전 정보: {feature['versions']}")
    print(f"평균 길이: {feature['avg_length']:.0f}자")

=== 카테고리별 특징 분석 ===

[AI]
주요 키워드: ['인공지능이', '의료', '진단에']
기술 용어: ['Chat']
버전 정보: []
평균 길이: 42자

[개발]
주요 키워드: ['프레임워크', '자바스크립트', 'react']
기술 용어: ['Django', 'React']
버전 정보: ['5.0']
평균 길이: 32자

[보안]
주요 키워드: ['보안', '대규모', '랜섬웨어']
기술 용어: []
버전 정보: []
평균 길이: 26자


## 학습 내용 정리

### 핵심 전처리 기법

| 기법 | 목적 | 주요 방법 |
|------|------|----------|
| 정규표현식 | 패턴 매칭 | `re.findall()`, `re.sub()` |
| 텍스트 정제 | 노이즈 제거 | HTML, URL, 특수문자 제거 |
| 토큰화 | 단위 분할 | 문장, 단어, 형태소 분리 |
| 정규화 | 표준화 | 대소문자, 숫자, 반복 처리 |
| 불용어 제거 | 중요 단어 추출 | 빈도 높은 무의미 단어 제거 |

### 정규표현식 주요 패턴

| 패턴 | 설명 | 예시 |
|------|------|------|
| `\d+` | 숫자 | 123, 456 |
| `[가-힣]+` | 한글 단어 | 안녕, 세계 |
| `[a-zA-Z]+` | 영문 단어 | Hello, World |
| `\s+` | 공백 | 공백, 탭, 줄바꿈 |
| `.+?` | 최소 매칭 | 욕심없는 매칭 |

### 실습 완료 체크리스트

✓ 정규표현식으로 패턴 추출  
✓ 텍스트 정제 및 노이즈 제거  
✓ 토큰화와 N-gram 생성  
✓ 텍스트 정규화 적용  
✓ 불용어 처리 및 키워드 추출  
✓ 뉴스 전처리 파이프라인 구축